# AoC 2024 Day 17 — Chronospatial Computer

**Python — a 3-bit virtual machine**

Puzzle: <https://adventofcode.com/2024/day/17>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

A tiny 3-bit computer. Three registers `A`, `B`, `C` hold arbitrarily large integers; the program is a flat list of 3-bit numbers read as alternating `(opcode, operand)` pairs, with an instruction pointer starting at 0 and advancing by 2.

Operands come in two flavours. A **literal** operand is just its own value. A **combo** operand means 0–3 literally, and 4/5/6 mean the contents of A/B/C.

The eight opcodes:

| # | name | effect |
|---|------|--------|
| 0 | `adv` | `A = A >> combo` (integer divide by `2**combo`) |
| 1 | `bxl` | `B ^= literal` |
| 2 | `bst` | `B = combo % 8` |
| 3 | `jnz` | if `A != 0`, set `ip = literal` and do **not** advance |
| 4 | `bxc` | `B ^= C` (reads an operand, ignores it) |
| 5 | `out` | emit `combo % 8` |
| 6 | `bdv` | `B = A >> combo` |
| 7 | `cdv` | `C = A >> combo` |

Reading an opcode past the end of the program halts the machine.

- **Part 1** — run the program from the given register values and join everything `out` emitted with commas. The answer is a **string**, not a number.

## The approach

A virtual machine is the archetypal anti-Spark workload, and it is worth naming why rather than hand-waving at "it's sequential".

Spark needs **a set of rows whose processing is independent**. A VM has the opposite shape: there is exactly one row — the machine state `(A, B, C, ip, output)` — and instruction *n+1* cannot even be *identified* until instruction *n* has run, because `jnz` decides the next instruction pointer from a register that the preceding instructions wrote. There is no partitioning key, no set to fan out over, and no way to speculate ahead. Parallelism of one.

You could technically express it as a DataFrame with a single row and loop jobs until halt — one Spark job per *instruction*, thousands of them, to move three integers around. The entire program is a handful of shifts, XORs and mod-8s; CPython runs the whole thing in well under a millisecond.

The one implementation detail worth calling out: **part 1 answers with a string**, not an int — `'7,3,5,7,5,7,4,3,0'`. The `out` instruction emits 3-bit values that the puzzle asks you to join with commas, so `part1` returns `str`. Anything downstream that assumes these days return integers will need to cope.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day17

spark = get_spark('aoc-2024-day17')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = 'Register A: 729\nRegister B: 0\nRegister C: 0\n\nProgram: 0,1,5,4,3,0\n'

print('part 1:', day17.part1(spark, EXAMPLE), '(expected 4,6,3,5,6,3,5,2,1,0)')

### The state dependency, one instruction at a time

Read the `ip` column downwards. It walks `0, 2, 4` and then **jumps back to 0** — and which way it jumps is decided by register `A`, which the instructions above it just modified. That backward edge is why nothing here can be scheduled ahead of time.

In [ ]:
OPS = ['adv', 'bxl', 'bst', 'jnz', 'bxc', 'out', 'bdv', 'cdv']
COMBO = {0: '0', 1: '1', 2: '2', 3: '3', 4: 'A', 5: 'B', 6: 'C', 7: '!'}

reg, program = day17.parse(EXAMPLE)
print('program:', ','.join(map(str, program)))
print('decoded: ', ' '.join(f'{OPS[program[i]]} {program[i + 1]}' for i in range(0, len(program) - 1, 2)))
print(f'start:    A={reg[0]} B={reg[1]} C={reg[2]}')
print()

# Step the VM by hand, printing state before each instruction. Note how ip
# jumps backwards: instruction n+1 is not knowable until n has executed.
print('step  ip   instr  combo      A   B   C  out')
ip, out, step = 0, [], 0
while ip + 1 < len(program) and step < 18:
    opcode, operand = program[ip], program[ip + 1]
    combo = operand if operand < 4 else reg[operand - 4]
    instr = OPS[opcode] + ' ' + str(operand)
    emitted = ','.join(map(str, out))
    print(f'{step:>4} {ip:>3} {instr:>7} {COMBO[operand]:>5} {reg[0]:>6} {reg[1]:>3} {reg[2]:>3}  {emitted}')
    step += 1
    if opcode == 0:
        reg[0] //= 2**combo
    elif opcode == 1:
        reg[1] ^= operand
    elif opcode == 2:
        reg[1] = combo % 8
    elif opcode == 3 and reg[0] != 0:
        ip = operand
        continue
    elif opcode == 4:
        reg[1] ^= reg[2]
    elif opcode == 5:
        out.append(combo % 8)
    elif opcode == 6:
        reg[1] = reg[0] // 2**combo
    elif opcode == 7:
        reg[2] = reg[0] // 2**combo
    ip += 2

answer = day17.part1(spark, EXAMPLE)
print()
print('part 1 returns', repr(answer), '-- a', type(answer).__name__ + ', not an int')

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 17)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day17.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day17 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- **Part 1 returns a string**, e.g. `'4,6,3,5,6,3,5,2,1,0'` for the example. Not an int, not a list. Comparing it to a number will silently be `False`.
- `parse` uses `re.findall(r'-?\d+')` across the *whole* input and takes the first three numbers as A, B, C and the rest as the program. That works because the AoC format puts the registers first and every remaining number belongs to the program — but it means the labels (`Register A:`, `Program:`) aren't actually checked. Reordered input would parse cleanly and give the wrong answer.
- **Literal vs combo operands is the classic bug.** `bxl` (1) and `jnz` (3) read their operand *literally*; `adv`/`bst`/`out`/`bdv`/`cdv` read it as a *combo* value, where 4/5/6 mean registers A/B/C. Using `combo` for `bxl` gets you a plausible-looking wrong answer.
- `jnz` uses `continue`, deliberately skipping the `ip += 2`. That is the whole point of a jump — the puzzle states the pointer is *not* advanced when the jump is taken.
- The `elif opcode == 3 and reg[0] != 0` chain is subtle: when `A == 0` the condition fails, so control falls past every other branch to `ip += 2` and the jump is correctly ignored. Restructuring that `if`/`elif` ladder into separate statements breaks it.
- Halting is `ip + 1 < len(program)` — reading an opcode *or* its operand past the end stops the machine. Registers are unbounded Python ints; only the program values and `out` results are 3-bit.
- `combo` operand 7 is reserved and never appears in a valid program, so `reg[operand - 4]` is never indexed out of range. An invalid program would raise `IndexError` rather than report anything useful.